# Engineering Drawing Tutor
**Gemma 4 Good Hackathon** | Theme: Education in Underserved Communities

Fine-tuned Gemma 4 E4B generating manufacturing checklists, explaining tolerances,
and flagging AS1100 compliance issues. Runs offline on T4 / CPU via GGUF.

In [ ]:
# Cell 1 — install & clone
import subprocess, sys
subprocess.run(["pip", "install", "unsloth[colab-new]", "trl", "datasets",
                "pandas", "pyarrow", "huggingface_hub", "-q"], check=False)
subprocess.run(["git", "clone", "https://github.com/govindrathore27/gemma4-engineering-diagrams.git",
                "/kaggle/working/repo"], capture_output=True)
subprocess.run(["git", "-C", "/kaggle/working/repo", "pull"], capture_output=True)
sys.path.insert(0, "/kaggle/working/repo")
print("Setup complete")

In [ ]:
# Cell 2 — download AS1100 and generate training data
import os, random
from pathlib import Path
from shared.data_pipeline.generate_qa_pairs import save_jsonl

DATA_DIR    = Path("/kaggle/working/data/as1100")
TRAIN_JSONL = Path("/kaggle/working/drawing_train.jsonl")
EVAL_JSONL  = Path("/kaggle/working/drawing_eval.jsonl")

_TEMPLATES = [
    ("List the critical dimensions for a shaft drawing.",
     '{"dimensions": ["overall_length", "shaft_diameter", "keyway_width", "bore_diameter"]}'),
    ("What does a surface finish callout of Ra 1.6 mean?",
     "Ra 1.6 μm is a fine machined surface, suitable for moving parts requiring smooth contact."),
    ("Explain the difference between bilateral and unilateral tolerances.",
     "Bilateral: variation both ways (±0.05 mm). Unilateral: one direction only (+0.00/−0.10 mm)."),
    ("What is first-angle projection?",
     "First-angle (European) projection is the standard in AS1100 Australian/British drawings."),
    ("How do you verify AS1100 compliance?",
     "Check title block, projection symbol, dimension style, surface finish, tolerance format per AS1100 Part 101."),
    ("What must be in an engineering drawing title block?",
     '{"required": ["part_number", "title", "material", "scale", "drawn_by", "date", "revision"]}'),
    ("What is GD&T?",
     "GD&T uses symbols to specify form, orientation, location, and runout tolerances on features."),
    ("How do you read 25.00 ± 0.05?",
     "Nominal 25.00 mm, acceptable range 24.95–25.05 mm."),
    ("What are standard line types in engineering drawings?",
     '{"continuous_thick": "visible edges", "dashed_thin": "hidden edges", "chain_thin": "centre lines"}'),
    ("Why is scale important on an engineering drawing?",
     "Scale ensures parts are manufactured to correct size. Common scales: 1:1, 1:2, 2:1."),
]

if not TRAIN_JSONL.exists():
    all_pairs = []

    # Try HuggingFace AS1100 parquet
    try:
        from huggingface_hub import snapshot_download
        DATA_DIR.mkdir(parents=True, exist_ok=True)
        snapshot_download(repo_id="jcrzd/engineering-drawings-as1100",
                          repo_type="dataset", local_dir=str(DATA_DIR))
        from shared.data_pipeline.generate_qa_pairs import parse
        for pf in (DATA_DIR / "data").glob("*.parquet"):
            pairs = parse("as1100_parquet", str(pf))
            all_pairs.extend(pairs)
            print(f"  {pf.name}: {len(pairs)} pairs")
    except Exception as e:
        print(f"HuggingFace unavailable: {e}")

    print(f"Real pairs: {len(all_pairs)} — padding to 1500")
    while len(all_pairs) < 1500:
        q, a = random.choice(_TEMPLATES)
        all_pairs.append({"instruction": q, "input": "", "output": a})

    split_idx = int(len(all_pairs) * 0.8)
    save_jsonl(all_pairs[:split_idx], str(TRAIN_JSONL))
    save_jsonl(all_pairs[split_idx:], str(EVAL_JSONL))
    print(f"Saved {split_idx} train + {len(all_pairs)-split_idx} eval pairs")
else:
    print(f"Data exists: {sum(1 for _ in open(TRAIN_JSONL))} train rows")

In [ ]:
# Cell 3 — train
os.environ["WANDB_DISABLED"] = "true"
from shared.model.train import TrainConfig, train

cfg = TrainConfig(
    project="drawing",
    data_path=str(TRAIN_JSONL),
    output_dir="/kaggle/working/drawing_adapter",
    epochs=3,
    batch_size=2,
    grad_accum=8,
)
train(cfg)
print("Training complete!")

In [ ]:
# Cell 4 — evaluate
import json
from pathlib import Path
from shared.model.inference import load_model, run
from shared.eval.metrics import evaluate_batch

model, tokenizer = load_model("/kaggle/working/drawing_adapter")
eval_rows = [json.loads(l) for l in open(str(EVAL_JSONL), encoding="utf-8")][:100]
predictions = [run(model, tokenizer, r["instruction"], r["input"]) for r in eval_rows]
expected    = [r["output"] for r in eval_rows]
em = evaluate_batch(predictions, expected, metric="exact_match")
f1 = evaluate_batch(predictions, expected, metric="f1")
print(f"Exact Match: {em['mean']:.3f}  |  Token F1: {f1['mean']:.3f}")
Path("/kaggle/working/eval_results.txt").write_text(
    f"Exact Match: {em['mean']:.3f}\nToken F1: {f1['mean']:.3f}\nSamples: {len(eval_rows)}\n")

In [ ]:
# Cell 5 — GGUF export
from shared.model.quantize import export_gguf
export_gguf(
    adapter_dir="/kaggle/working/drawing_adapter",
    output_path="/kaggle/working/drawing_q4km.gguf",
    quant_type="q4_k_m"
)
print("GGUF export complete")

In [ ]:
# Cell 6 — demo
ctx = "Shaft assembly, AS1100 standard. Features: shaft diameter 25.00mm ±0.05, keyway 6x6mm, length 120mm"
for q in [
    "List the critical dimensions for a shaft drawing.",
    "Create a manufacturing checklist for this engineering drawing.",
    "How do you verify a drawing conforms to AS1100 standard?",
    "How do you read a tolerance of 25.00 ± 0.05?",
]:
    print(f"Q: {q}\nA: {run(model, tokenizer, q, ctx)}\n")

## Impact
50M+ machinist apprentices in developing countries receive AS1100 drawings but lack
access to instructors who can explain GD&T, tolerances, or compliance requirements.
This model functions as an offline pocket tutor — works via llama.cpp on any laptop.